In [ ]:
dir.create("../../results/diablo/figures/", showWarnings = FALSE, recursive = TRUE)
options(repr.plot.width = 9, repr.plot.height = 6, repr.plot.res = 150)
knitr::opts_chunk$set(warning = FALSE, message = FALSE)

## Overview

Before building a multi-omics classifier, we establish **single-omics baselines**
using sparse Partial Least Squares Discriminant Analysis (sPLS-DA) — the single-block
version of DIABLO. This answers:

1. How well can each omics layer classify PAM50 subtypes independently?
2. Which layer is most informative?
3. What does the multi-omics model need to improve upon?

If a single omics layer already achieves near-perfect classification, the case for
integration is weaker. In practice, each layer adds complementary information, and
the integrated model outperforms all single-omic baselines — which we confirm in Notebook 3.

**Method**: `mixOmics::splsda()` with M-fold cross-validation via `perf()`.
The **Balanced Error Rate (BER)** is the primary metric: it accounts for class
imbalance by averaging per-class error rates.

---

## Setup

In [ ]:
library(mixOmics)
library(ggplot2)
library(dplyr)
library(tidyr)
library(purrr)
library(pheatmap)

theme_set(theme_bw(base_size = 12))

subtype_colors <- c(Basal = "#E41A1C", Her2 = "#FF7F00", LumA = "#4DAF4A")

saved   <- readRDS("../../results/diablo/breast_TCGA_processed.RDS")
X_train <- saved$X_train
Y_train <- saved$Y_train
X_test  <- saved$X_test
Y_test  <- saved$Y_test

cat("Data loaded.\n")

---

## 1. Tune and Train sPLS-DA Per Block

In [ ]:
# Helper function: for a single omics block, tune the number of components
# and keepX parameter, then train the final sPLS-DA model.
# Returns the trained model and its cross-validated BER.

run_splsda_baseline <- function(X, Y, block_name,
                                 n_comp    = 3,
                                 test_keepX = c(5, 10, 15, 20, 25, 30),
                                 folds     = 10,
                                 nrepeat   = 5) {
  cat(sprintf("\n--- %s ---\n", block_name))

  # Step 1: Basic sPLS-DA with all features to get ncomp estimate
  model_basic <- splsda(X, Y, ncomp = n_comp, keepX = rep(ncol(X), n_comp))

  # Step 2: Tune keepX via cross-validation
  set.seed(42)
  tune_res <- tune.splsda(
    X           = X,
    Y           = Y,
    ncomp       = n_comp,
    validation  = "Mfold",
    folds       = folds,
    nrepeat     = nrepeat,
    test.keepX  = test_keepX,
    dist        = "mahalanobis.dist",
    progressBar = FALSE
  )

  optimal_keepX <- tune_res$choice.keepX
  cat("  Optimal keepX:", paste(optimal_keepX, collapse=", "), "\n")

  # Step 3: Train final sparse model
  model_final <- splsda(X, Y, ncomp = n_comp, keepX = optimal_keepX)

  # Step 4: Cross-validate the final sparse model
  set.seed(42)
  cv_res <- perf(
    model_final,
    validation  = "Mfold",
    folds       = folds,
    nrepeat     = nrepeat,
    progressBar = FALSE
  )

  # Extract BER for mahalanobis distance at optimal ncomp
  ber_by_comp <- cv_res$error.rate$BER[, "mahalanobis.dist"]
  optimal_ncomp <- which.min(ber_by_comp)
  min_ber       <- min(ber_by_comp, na.rm = TRUE)

  cat(sprintf("  Optimal ncomp: %d | Min BER: %.3f\n", optimal_ncomp, min_ber))

  list(
    block         = block_name,
    model         = model_final,
    cv            = cv_res,
    tune          = tune_res,
    optimal_keepX = optimal_keepX,
    optimal_ncomp = optimal_ncomp,
    min_ber       = min_ber
  )
}

In [ ]:
# Check for cached results to avoid re-running time-consuming CV
baseline_path <- "../../results/diablo/single_omics_baselines.RDS"

if (!file.exists(baseline_path)) {
  baselines <- map2(
    X_train, names(X_train),
    ~ run_splsda_baseline(.x, Y_train, block_name = .y)
  )
  saveRDS(baselines, baseline_path)
  cat("\nBaseline models saved.\n")
} else {
  baselines <- readRDS(baseline_path)
  cat("Pre-computed baseline models loaded.\n")
}

---

## 2. Cross-Validated BER Curves per Block

In [ ]:
# Plot BER vs number of components for each block.
# The optimal ncomp (elbow) shows where adding more components
# no longer reduces classification error.

ber_df <- map_dfr(baselines, function(bl) {
  ber <- bl$cv$error.rate$BER[, "mahalanobis.dist"]
  data.frame(
    block = bl$block,
    ncomp = seq_along(ber),
    BER   = ber
  )
})

ggplot(ber_df, aes(x = ncomp, y = BER, colour = block)) +
  geom_line(linewidth = 0.9) +
  geom_point(size = 2.5) +
  scale_colour_brewer(palette = "Set1") +
  scale_y_continuous(labels = scales::percent) +
  labs(
    title    = "Single-omics sPLS-DA: BER vs number of components",
    subtitle = "Lower BER = better classification. Optimal ncomp at curve elbow.",
    x        = "Number of components", y = "Balanced Error Rate (CV)",
    colour   = "Omics block"
  ) +
  theme(legend.position = "right")

---

## 3. Minimum BER Comparison

In [ ]:
# Bar plot of minimum BER per block — the single-omics classification
# performance ceiling. The multi-omics model should improve on all of these.

ber_summary <- map_dfr(baselines, ~ data.frame(
  block   = .x$block,
  min_BER = .x$min_ber,
  ncomp   = .x$optimal_ncomp
))

ggplot(ber_summary, aes(x = reorder(block, min_BER), y = min_BER, fill = block)) +
  geom_col(show.legend = FALSE, alpha = 0.85) +
  geom_text(
    aes(label = sprintf("BER=%.1f%%\n(ncomp=%d)", 100 * min_BER, ncomp)),
    hjust = -0.1, size = 3.5
  ) +
  coord_flip() +
  scale_y_continuous(labels = scales::percent, limits = c(0, 0.6)) +
  scale_fill_brewer(palette = "Set1") +
  labs(
    title    = "Single-omics minimum BER (10-fold CV, 5 repeats)",
    subtitle = "Baseline for DIABLO multi-omics model to beat",
    x        = NULL, y = "Balanced Error Rate"
  )

# Print summary table
cat("\n=== Single-omics baseline summary ===\n")
print(ber_summary)

---

## 4. Sample Score Plots per Block

In [ ]:
# sPLS-DA component score plots show how well each single omic separates subtypes.
# Overlapping ellipses = poor discrimination; non-overlapping = good.

plotIndiv(
  baselines$mRNA$model,
  comp         = c(1, 2),
  ind.names    = FALSE,
  group        = Y_train,
  col.per.group = subtype_colors,
  ellipse      = TRUE,
  title        = "mRNA sPLS-DA: sample scores (Comp 1 vs 2)"
)

In [ ]:
plotIndiv(
  baselines$miRNA$model,
  comp          = c(1, 2),
  ind.names     = FALSE,
  group         = Y_train,
  col.per.group  = subtype_colors,
  ellipse       = TRUE,
  title         = "miRNA sPLS-DA: sample scores (Comp 1 vs 2)"
)

In [ ]:
plotIndiv(
  baselines$proteomics$model,
  comp          = c(1, 2),
  ind.names     = FALSE,
  group         = Y_train,
  col.per.group  = subtype_colors,
  ellipse       = TRUE,
  title         = "Proteomics sPLS-DA: sample scores (Comp 1 vs 2)"
)

---

## 5. Loading Plots per Block

In [ ]:
# Loading plots show which features define the discriminant components.
# Colour = subtype where the feature is most highly expressed.
# Features at the extremes are the strongest discriminators.

plotLoadings(
  baselines$mRNA$model,
  comp    = 1,
  contrib = "max",
  method  = "mean",
  title   = "mRNA sPLS-DA: feature loadings — Component 1",
  col.per.group = subtype_colors
)

In [ ]:
plotLoadings(
  baselines$miRNA$model,
  comp    = 1,
  contrib = "max",
  method  = "mean",
  title   = "miRNA sPLS-DA: feature loadings — Component 1",
  col.per.group = subtype_colors
)

---

## 6. Test Set Accuracy per Block

In [ ]:
# Evaluate each single-omics model on the held-out test set.
# NOTE: proteomics test data is not available in this dataset split;
#       those rows will be NA and excluded from the accuracy summary.

test_results <- map_dfr(baselines, function(bl) {
  test_block <- X_test[[bl$block]]

  # Skip if block not present in test set
  if (is.null(test_block)) {
    return(data.frame(
      block       = bl$block,
      overall_acc = NA_real_,
      Basal_acc   = NA_real_,
      Her2_acc    = NA_real_,
      LumA_acc    = NA_real_
    ))
  }

  test_pred <- predict(
    bl$model,
    newdata = test_block,
    dist    = "mahalanobis.dist"
  )

  preds   <- test_pred$class$mahalanobis.dist[, bl$optimal_ncomp]
  correct <- preds == as.character(Y_test)

  per_class <- tapply(correct, Y_test, mean, na.rm = TRUE)

  data.frame(
    block       = bl$block,
    overall_acc = mean(correct, na.rm = TRUE),
    Basal_acc   = per_class["Basal"],
    Her2_acc    = per_class["Her2"],
    LumA_acc    = per_class["LumA"]
  )
})

cat("=== Test set accuracy per single-omics model ===\n")
print(test_results |> mutate(across(where(is.numeric), ~ round(.x * 100, 1))))

---

## 7. Save Baselines

In [ ]:
# Save individual model performance for comparison with DIABLO in Notebook 3
saveRDS(
  list(baselines = baselines, test_results = test_results),
  "../../results/diablo/single_omics_results.RDS"
)

cat("Baseline results saved to results/diablo/single_omics_results.RDS\n")

---

## Summary

In [ ]:
cat("=== Single-omics baseline summary ===\n\n")

ber_summary |>
  left_join(
    test_results |> select(block, overall_acc),
    by = "block"
  ) |>
  mutate(
    min_BER_pct  = round(100 * min_BER, 1),
    test_acc_pct = round(100 * overall_acc, 1)
  ) |>
  select(block, ncomp, min_BER_pct, test_acc_pct) |>
  print()

**Key interpretation**:
- Each single-omics model achieves meaningful classification (BER < 20%), confirming
  that PAM50 signal is present across all three layers
- No single layer dominates completely — all three contribute unique discriminant information
- The multi-omics DIABLO model in Notebook 3 should achieve lower BER than any single block
  by combining complementary signals from mRNA regulation (miRNA), expression (mRNA), and
  downstream functional activity (proteomics)

**The case for integration**: Orthogonal regulatory layers (transcription, post-transcriptional
regulation, protein activity) carry non-redundant information. DIABLO's design matrix
explicitly exploits cross-block correlations to build a more robust discriminant.

**→ Next: `03_diablo_integration.Rmd`**